## Modelling prechecks

**Objective:** Verify two assumptions flagged in `modelling_decisions.md` before feature extraction begins, so neither assumption is silently carried into the pipeline unchecked.

1. **Protocol composition vs IAF-proximity (gates decision 5).** IAF-proximity-to-10Hz replicated in Roelofs et al. (2021) only for the 10Hz unilateral left-DLPFC subgroup, not for 1Hz or bilateral protocols. This check establishes the protocol composition of the 163-subject cohort and whether protocol is itself confounded with responder status, to determine whether IAF-proximity can be used cohort-wide, needs restricting to a subgroup, or needs protocol as a covariate.

2. **Age vs retained-epoch-count (gates decision 3).** Age is an established confound on responder status (younger subjects respond more, Welch's t=-2.68, p=.01). The ~10-epoch inclusion floor was set on autoreject-stability grounds, not tested against age. This check establishes whether epoch retention correlates with age, which would mean the floor systematically excludes subjects by age and deepens rather than controls the confound.

**Inputs:**
- Check 1: `data/cohort_filtered_n163.xlsx` (full 163-subject cohort, protocol field, responder label)
- Check 2: QC metadata from the full 163-subject preprocessing run (`04_full_cohort_run.ipynb`)

**Assumptions / limitations stated upfront:**
- Check 1 runs on the full 163-subject cohort as defined by clinical/label criteria.
- Check 2 runs on the 160 subjects with usable QC metadata from the full preprocessing run
  (`04_full_cohort_run.ipynb`, logged in `data/batch_results_log_full_cohort.csv`). Three subjects
  in the 163-subject cohort have no raw BDF file on disk and are excluded here as a data
  availability gap, not a preprocessing failure, worth flagging if either excluded subject
  distribution turns out responder/non-responder imbalanced.
- Check 2 uses one row per subject (restEC condition, heog_off variant, the primary dataset
  variant per decision 1) to avoid double-counting across conditions/variants.

In [1]:
# imports and setup
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import importlib
import src.preprocessing
importlib.reload(src.preprocessing)
from src.preprocessing import *

In [2]:
#Cohort loading 
data_dir = find_repo_root() / "data"

cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")
all_subject_ids = cohort_df['TDBRAIN_ID'].tolist()
print(f"{len(all_subject_ids)} subjects in cohort")

163 subjects in cohort


In [3]:
# Check available columns before assuming a specific protocol-field name 
# TDBRAIN's spreadsheet may label this stim frequency/hemisphere field
# differently than expected (e.g. 'stim_protocol', 'location', 'frequency')
print(cohort_df.columns.tolist())

['TDBRAIN_ID', 'DISC/REP', 'indication', 'formal_status', 'Dataset', 'Consent', 'sessSeason', 'sessTime', 'Responder', 'Remitter', 'age', 'gender', 'sessID', 'nrSessions', 'neoFFI_q1', 'neoFFI_q2', 'neoFFI_q3', 'neoFFI_q4', 'neoFFI_q5', 'neoFFI_q6', 'neoFFI_q7', 'neoFFI_q8', 'neoFFI_q9', 'neoFFI_q10', 'neoFFI_q11', 'neoFFI_q12', 'neoFFI_q13', 'neoFFI_q14', 'neoFFI_q15', 'neoFFI_q16', 'neoFFI_q17', 'neoFFI_q18', 'neoFFI_q19', 'neoFFI_q20', 'neoFFI_q21', 'neoFFI_q22', 'neoFFI_q23', 'neoFFI_q24', 'neoFFI_q25', 'neoFFI_q26', 'neoFFI_q27', 'neoFFI_q28', 'neoFFI_q29', 'neoFFI_q30', 'neoFFI_q31', 'neoFFI_q32', 'neoFFI_q33', 'neoFFI_q34', 'neoFFI_q35', 'neoFFI_q36', 'neoFFI_q37', 'neoFFI_q38', 'neoFFI_q39', 'neoFFI_q40', 'neoFFI_q41', 'neoFFI_q42', 'neoFFI_q43', 'neoFFI_q44', 'neoFFI_q45', 'neoFFI_q46', 'neoFFI_q47', 'neoFFI_q48', 'neoFFI_q49', 'neoFFI_q50', 'neoFFI_q51', 'neoFFI_q52', 'neoFFI_q53', 'neoFFI_q54', 'neoFFI_q55', 'neoFFI_q56', 'neoFFI_q57', 'neoFFI_q58', 'neoFFI_q59', 'neoFFI_q60

In [5]:
# Inspect raw values before crosstabbing - clinical spreadsheet fields often have
# inconsistent formatting (spacing, capitalisation) or missing values that would
# silently fragment the crosstab into spurious extra categories if not caught first
print(cohort_df['rTMS PROTOCOL'].value_counts(dropna=False))

rTMS PROTOCOL
2.0    85
1.0    42
3.0    32
NaN     4
Name: count, dtype: int64


In [6]:
# Check whether the 7-vs-32 discrepancy against van Dijk et al.'s Table 2 is a
# denominator artefact (their "7" was counted under a stricter BDI pre&post
# filter) or a genuine dataset-version difference between the 2022 paper and
# our V3.1 release. Filtering to subjects with both BDI scores present should
# reproduce something close to Table 2's counts (65 / 105 / 7) if it's just
# a denominator issue - if the gap persists, it's a real V3.1 vs. paper difference.
bdi_complete = cohort_df[cohort_df['BDI_pre'].notna() & cohort_df['BDI_post'].notna()]

print(f"{len(bdi_complete)} subjects with complete BDI pre/post (of {len(cohort_df)} total)")
print("\nProtocol counts, BDI-complete subjects only:")
print(bdi_complete['rTMS PROTOCOL'].value_counts(dropna=False))

print("\nProtocol counts, full 163-subject cohort (for comparison):")
print(cohort_df['rTMS PROTOCOL'].value_counts(dropna=False))

163 subjects with complete BDI pre/post (of 163 total)

Protocol counts, BDI-complete subjects only:
rTMS PROTOCOL
2.0    85
1.0    42
3.0    32
NaN     4
Name: count, dtype: int64

Protocol counts, full 163-subject cohort (for comparison):
rTMS PROTOCOL
2.0    85
1.0    42
3.0    32
NaN     4
Name: count, dtype: int64


In [7]:
# Check whether responder status differs by protocol - if response rates diverge
# by protocol, that's a second confound sitting alongside age (Decision 7), and
# it matters here specifically because protocol is already being used to gate
# which subjects get the IAF-proximity feature (Decision 5). A crosstab with
# normalized rows shows response rate per protocol directly.
protocol_response = pd.crosstab(cohort_df['rTMS PROTOCOL'], cohort_df['Responder'])
protocol_response_pct = pd.crosstab(cohort_df['rTMS PROTOCOL'], cohort_df['Responder'], normalize='index')

print("Counts:")
print(protocol_response)
print("\nResponse rate by protocol:")
print(protocol_response_pct)

# Chi-square test for independence: is responder status associated with protocol?
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(protocol_response.dropna())
print(f"\nChi-square test: chi2={chi2:.2f}, p={p:.3f}")

Counts:
Responder       0   1
rTMS PROTOCOL        
1.0            17  25
2.0            31  54
3.0            21  11

Response rate by protocol:
Responder             0         1
rTMS PROTOCOL                    
1.0            0.404762  0.595238
2.0            0.364706  0.635294
3.0            0.656250  0.343750

Chi-square test: chi2=8.24, p=0.016


In [8]:
# Check whether protocol is also associated with age - if younger subjects are
# concentrated in specific protocols, the age confound (Decision 7) and this
# protocol/response association may not be independent
print(cohort_df.groupby('rTMS PROTOCOL')['age'].describe()[['mean', 'std', 'count']])

from scipy.stats import f_oneway
groups = [cohort_df[cohort_df['rTMS PROTOCOL']==p]['age'].dropna() for p in [1.0, 2.0, 3.0]]
f_stat, p_val = f_oneway(*groups)
print(f"\nOne-way ANOVA (age by protocol): F={f_stat:.2f}, p={p_val:.3f}")

                    mean        std  count
rTMS PROTOCOL                             
1.0            39.904048  12.302029   42.0
2.0            46.140706  14.313157   85.0
3.0            50.328125  13.355298   32.0

One-way ANOVA (age by protocol): F=5.64, p=0.004


### Check 1 - Findings

**Protocol composition (n=163):** Protocol 1 (10Hz left-DLPFC) = 42, Protocol 2
(1Hz right-DLPFC) = 85, Protocol 3 (undefined, see `modelling_decisions.md`) = 32,
NaN = 4.

**Protocol 3 vs. published counts:** van Dijk et al. (2022) Table 2 reports 7
protocol-3 subjects database-wide; this cohort shows 32. Not a denominator artefact
(all 163 have complete BDI pre/post). Cause unresolved, documented as an open item.

**Protocol is associated with responder status** (chi-square = 8.24, p = 0.016).
Response rate: Protocol 1 = 59.5%, Protocol 2 = 63.5%, Protocol 3 = 34.4%.

**Protocol is also associated with age** (F = 5.64, p = 0.004): mean age rises with
protocol number (39.9 / 46.1 / 50.3). Protocol 3's lower response rate may be
partly attributable to age rather than protocol itself — not established here,
deferred to the fold-nested age regression (Decision 7).

**Downstream decision:** IAF-proximity's literature support only covers protocol 1
(n=42), so it's excluded from the primary pool and reported as a supplementary
analysis on protocol-1 subjects only (Decision 5). Protocol 2, 3, and NaN (n=4)
subjects are excluded from that arm; all remain in the primary and secondary arms.

In [9]:
# Confirm column names (epoch counts, condition/variant labels, subject ID) and
# check row count matches expectations (160 subjects x 2 conditions x 2 HEOG
# variants = 640 rows) before filtering, since silently filtering on a wrong
# column name would fail quietly rather than throwing an error.
qc_log = pd.read_csv(data_dir / "batch_results_log_full_cohort.csv")

print(f"{len(qc_log)} rows, {qc_log['subject_id'].nunique() if 'subject_id' in qc_log.columns else '?'} unique subjects")
print(qc_log.columns.tolist())
qc_log.head()

652 rows, 163 unique subjects
['subject_id', 'condition', 'heog_variant', 'status', 'n_epochs_before', 'n_epochs_after', 'output_path', 'autoreject_consensus', 'autoreject_n_interpolate', 'autoreject_extreme', 'heog_n_candidates', 'heog_n_valid', 'heog_correction_applied', 'error']


,subject_id,condition,heog_variant,status,n_epochs_before,n_epochs_after,output_path,autoreject_consensus,autoreject_n_interpolate,autoreject_extreme,heog_n_candidates,heog_n_valid,heog_correction_applied,error
0,sub-87999321,restEC,heog_off,ok,24.0,23.0,/Users/romyweinstock/eeg-rtms-response-predict...,0.2,4.0,False,4.0,0.0,False,NaN
1,sub-87999321,restEO,heog_off,ok,24.0,22.0,/Users/romyweinstock/eeg-rtms-response-predict...,0.2,4.0,False,78.0,15.0,False,NaN
2,sub-87999321,restEC,heog_on,ok,24.0,23.0,/Users/romyweinstock/eeg-rtms-response-predict...,0.2,4.0,False,4.0,0.0,True,NaN
3,sub-87999321,restEO,heog_on,ok,24.0,22.0,/Users/romyweinstock/eeg-rtms-response-predict...,0.2,4.0,False,78.0,15.0,True,NaN
4,sub-88049537,restEC,heog_off,ok,24.0,23.0,/Users/romyweinstock/eeg-rtms-response-predict...,1.0,25.0,True,66.0,7.0,False,NaN


In [10]:
# Confirm status values before filtering to check whether the 3
# subjects with missing BDF files (found earlier in 04_full_cohort_run.ipynb)
# appear here with a non-'ok' status, since including them with null/garbage
# epoch counts would corrupt the age correlation below.
print(qc_log['status'].value_counts(dropna=False))

missing_subjects = ['sub-88026321', 'sub-88022133', 'sub-88041397']
print(qc_log[qc_log['subject_id'].isin(missing_subjects)][['subject_id', 'condition', 'heog_variant', 'status', 'n_epochs_after', 'error']])

status
ok       640
error     12
Name: count, dtype: int64
       subject_id condition heog_variant status  n_epochs_after  \
640  sub-88026321    restEC     heog_off  error             NaN   
641  sub-88026321    restEO     heog_off  error             NaN   
642  sub-88026321    restEC      heog_on  error             NaN   
643  sub-88026321    restEO      heog_on  error             NaN   
644  sub-88022133    restEC     heog_off  error             NaN   
645  sub-88022133    restEO     heog_off  error             NaN   
646  sub-88022133    restEC      heog_on  error             NaN   
647  sub-88022133    restEO      heog_on  error             NaN   
648  sub-88041397    restEC     heog_off  error             NaN   
649  sub-88041397    restEO     heog_off  error             NaN   
650  sub-88041397    restEC      heog_on  error             NaN   
651  sub-88041397    restEO      heog_on  error             NaN   

                                                 error  
640  The fil

In [11]:
# Filter to one row per subject: restEC condition, heog_off variant (primary
# dataset variant per Decision 1), status == 'ok' only (excludes the 3 subjects
# with missing BDF files). This avoids double-counting subjects across the
# 4 condition/variant combinations and excludes rows with no real epoch data.
epoch_data = qc_log[
    (qc_log['condition'] == 'restEC') &
    (qc_log['heog_variant'] == 'heog_off') &
    (qc_log['status'] == 'ok')
][['subject_id', 'n_epochs_after']]

print(f"{len(epoch_data)} subjects with valid restEC/heog_off epoch counts")

# Merge with age from the cohort spreadsheet. cohort_df uses 'TDBRAIN_ID',
# qc_log uses 'subject_id' - these should match once formatting is checked.
print("\nSample IDs, cohort_df:", cohort_df['TDBRAIN_ID'].head(3).tolist())
print("Sample IDs, qc_log:", epoch_data['subject_id'].head(3).tolist())

160 subjects with valid restEC/heog_off epoch counts

Sample IDs, cohort_df: ['sub-87999321', 'sub-88049537', 'sub-88049857']
Sample IDs, qc_log: ['sub-87999321', 'sub-88049537', 'sub-88049857']


In [12]:
# Merge epoch counts with age from cohort_df. Inner join by design: only
# subjects present in both (i.e. the 160 with valid QC data) are kept, so the
# 3 missing-BDF subjects are dropped automatically rather than appearing with
# NaN age or NaN epoch count.
merged = epoch_data.merge(
    cohort_df[['TDBRAIN_ID', 'age']],
    left_on='subject_id', right_on='TDBRAIN_ID', how='inner'
)

print(f"{len(merged)} subjects after merge")
print(merged[['subject_id', 'age', 'n_epochs_after']].describe())

# Correlate age against retained epoch count. Spearman is used rather than
# Pearson because epoch count is a bounded count variable (not necessarily
# linearly related to age), and Spearman only assumes a monotonic relationship,
# which is the weaker, more appropriate assumption here.
from scipy.stats import spearmanr, pearsonr
rho, p_spearman = spearmanr(merged['age'], merged['n_epochs_after'])
r, p_pearson = pearsonr(merged['age'], merged['n_epochs_after'])

print(f"\nSpearman: rho={rho:.3f}, p={p_spearman:.3f}")
print(f"Pearson: r={r:.3f}, p={p_pearson:.3f}")

160 subjects after merge
              age  n_epochs_after
count  160.000000      160.000000
mean    45.203500       23.156250
std     14.024705        0.555667
min     18.660000       21.000000
25%     34.900000       23.000000
50%     46.005000       23.000000
75%     54.572500       23.000000
max     78.000000       24.000000

Spearman: rho=-0.079, p=0.322
Pearson: r=-0.116, p=0.145


In [13]:
# Check the actual epoch-count distribution relative to the ~10-epoch floor 
# confirming whether any subjects are genuinely near the floor at all, since
# that changes how much the floor decision actually matters in practice.
print(merged['n_epochs_after'].value_counts().sort_index())
print(f"\nSubjects below 15 epochs: {(merged['n_epochs_after'] < 15).sum()}")
print(f"Subjects below 12 epochs: {(merged['n_epochs_after'] < 12).sum()}")

n_epochs_after
21.0      2
22.0      8
23.0    113
24.0     37
Name: count, dtype: int64

Subjects below 15 epochs: 0
Subjects below 12 epochs: 0


### Check 2 - Findings

**Sample:** 160 subjects with valid restEC/heog_off epoch counts (merged with age
from `cohort_filtered_n163.xlsx`). The 3 subjects with missing BDF files are
excluded via inner join, consistent with Check 1.

**No association between age and retained-epoch-count.** Spearman rho = -0.079
(p = 0.322); Pearson r = -0.116 (p = 0.145). Neither approaches significance. No
evidence that the ~10-epoch inclusion floor (Decision 3) introduces an age-related
selection effect.

**The floor is non-binding in this cohort.** Retained epochs range from 21 to 24
(min = 21, well above the 10-epoch floor); no subjects fall below 15, let alone
10. The floor was a necessary methodological decision to justify in principle, but
in practice no subjects in this cohort are excluded or marginal because of it.